In [1]:
import os
import torch
import numpy as np
from astropy.table import Table as aT
from astropy.table import join as aTjoin

In [2]:
from sedflow import flows as F

/global/homes/c/chahah/.conda/envs/gqp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import corner as DFM
# --- plotting ---
import matplotlib as mpl
import matplotlib.pyplot as plt
#mpl.rcParams['text.usetex'] = True
#mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

In [6]:
# selected LRG objects that Manu and Bernardita
columns = ['TARGETID', 'RA', 'DEC', "Z", "FLUX_G", "FLUX_R", "FLUX_Z", "FLUX_W1", "FLUX_W2", 
           "MW_TRANSMISSION_G", "MW_TRANSMISSION_R", "MW_TRANSMISSION_Z", "MW_TRANSMISSION_W1", "MW_TRANSMISSION_W2",
           "FLUX_IVAR_G", "FLUX_IVAR_R", "FLUX_IVAR_Z", "FLUX_IVAR_W1", "FLUX_IVAR_W2"]

lrgs = aT.read('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1/lrg_act/overlapping_catalog_legacy_info.txt', 
               format='ascii')

In [7]:
lrgs[:5]

col1,col2,col3,col4,col5,col6,col7,col8,col9,col10,col11,col12,col13,col14,col15,col16,col17,col18,col19
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
3.962782485577819e+16,231.78983150368694,1.5756522730323022,0.4000010129522464,2.6527600288391113,14.904266357421875,36.99319076538086,72.33859252929688,47.0587043762207,0.871864914894104,0.911770761013031,0.9496463537216187,0.9921806454658508,0.9951906204223633,431.7064208984375,109.5723648071289,14.836113929748535,1.8348532915115356,0.5141376852989197
3.962805710278196e+16,124.6405405128398,11.304746233552256,0.4000023511848214,3.2763376235961914,17.4721622467041,44.184120178222656,97.83049011230469,70.62666320800781,0.9270247220993042,0.9502375721931458,0.9718525409698486,0.9956713318824768,0.9973393678665161,371.2768249511719,101.56843566894531,18.065113067626953,1.3459391593933105,0.393652081489563
3.962793251936951e+16,179.39949401205334,6.043975493078079,0.40000245705986975,3.2347002029418945,17.660886764526367,41.50340270996094,74.24871063232422,51.024864196777344,0.9643861651420593,0.975868284702301,0.9864292144775391,0.9979260563850403,0.9987258315086365,412.82037353515625,90.31932830810547,28.332639694213867,1.7348061800003052,0.5064566731452942
3.962810636069013e+16,246.1506768162061,13.24783038101107,0.40000321582496334,2.0542750358581543,8.68364143371582,22.4169979095459,62.93301773071289,44.69700622558594,0.8629818558692932,0.9055026769638062,0.9459890723228455,0.9915990829467773,0.9948323369026184,248.93502807617188,93.3042221069336,19.55246353149414,2.0233192443847656,0.5988481640815735
3.962784811738905e+16,178.3301934146443,2.5214840636162226,0.40000370010623176,1.9931480884552002,10.557059288024902,31.65026092529297,72.83441925048828,47.33821105957031,0.9358174800872803,0.9562994837760925,0.975315511226654,0.9962095618247986,0.9976704716682434,252.413330078125,89.0908203125,24.748136520385742,1.4026813507080078,0.39119642972946167


In [9]:
# 1. correct for extinction
flux_g = lrgs['col5'] / lrgs['col10']
flux_r = lrgs['col6'] / lrgs['col11']
flux_z = lrgs['col7'] / lrgs['col12']
flux_w1 = lrgs['col8'] / lrgs['col13']
flux_w2 = lrgs['col9'] / lrgs['col14']

fluxes = np.array([flux_g, flux_r, flux_z, flux_w1, flux_w2]).T

# 2. compile other measurements
sig_g  = lrgs['col15']**-0.5
sig_r  = lrgs['col16']**-0.5
sig_z  = lrgs['col17']**-0.5
sig_w1 = lrgs['col18']**-0.5
sig_w2 = lrgs['col19']**-0.5

sig_fluxes = np.array([sig_g, sig_r, sig_z, sig_w1, sig_w2]).T

redshift = lrgs['col4']

/tmp/ipykernel_1469646/4078594948.py:11: RuntimeWarning: divide by zero encountered in power
  sig_g  = lrgs['col15']**-0.5


## load `DESIflow`

In [10]:
if torch.cuda.is_available(): device = 'cuda'
else: device = 'cpu'

desiflow = F.DESIflow(name='modelb.lowz.grzW1W2', device=device)

In [12]:
n_sample = 10000
posterior_samples = np.empty((len(lrgs), n_sample, 15))
for i in range(5): #len(bgs)):    
    posterior_samples[i,:,:] = desiflow.run(fluxes[i], sig_fluxes[i], redshift[i], Nsample=n_sample, progress_bar=False)

In [13]:
%timeit posterior_samples[i,:,:] = desiflow.run(fluxes[i], sig_fluxes[i], redshift[i], Nsample=n_sample, progress_bar=False)

156 ms ± 190 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
